In [3]:
import requests
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = "https://yuriafarm.bi-serv.net"



def get_token(login: str, password: str) -> str:
    
    url = f"{BASE_URL}/getToken?login={login}&password={password}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    try:
        return r.json()["token"]
    except:
        return r.text.strip()



def request_with_retry(url, retries=3, delay=2):

    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=60)

            if r.status_code == 200:
                return r.json()

            raise Exception(f"HTTP {r.status_code}")

        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(delay * (attempt + 1))



# Pagination
def fetch_route(route, token, dtStart=None, dtEnd=None):

    page = 1
    all_rows = []

    while True:

        if dtStart and dtEnd:
            url = (
                f"{BASE_URL}/api/reports/{route}"
                f"?dtStart={dtStart}&dtEnd={dtEnd}"
                f"&page={page}&token={token}"
            )
        else:
            url = (
                f"{BASE_URL}/api/reports/{route}"
                f"?page={page}&token={token}"
            )

        data = request_with_retry(url)

        if not data:
            break

        if isinstance(data, dict) and "data" in data:
            data = data["data"]

        if len(data) == 0:
            break

        all_rows.extend(data)
        page += 1

    return route, all_rows


def fetch_reports(
    login,
    password,
    routes,
    dtStart,
    dtEnd,
    max_workers=4
):

    if isinstance(routes, str):
        routes = [routes]

    token = get_token(login, password)

    results = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:

        futures = [
            executor.submit(fetch_route, r, token, dtStart, dtEnd)
            for r in routes
        ]

        for f in as_completed(futures):

            route, rows = f.result()

            if rows:
                df = pd.DataFrame(rows)
            else:
                df = pd.DataFrame()

            results[route] = df

    if len(results) == 1:
        return list(results.values())[0]

    return results

In [7]:
dfs = fetch_reports(
    login="yuriafarm@bi-serv.com&",
    password="Ciftas-6kovhi-cuqruh",
    routes=['buyers', 'products', 'outlets', 'emp'],
    dtStart= None,#"2026-01-01",
    dtEnd=None, #2026-01-02",
    max_workers=4
)

In [8]:
# save each dataframe in dfs to separate CSV files
for name, df in dfs.items():
    df.to_csv(f"{name}.csv", index=False)
print("saved dfs to stocks.csv and sales.csv")

saved dfs to stocks.csv and sales.csv


In [ ]:
import pandas as pd

TECH_COLS = ['TechExecutorRunID', 'TechProcessorRunID', 'TechProcessingDateTime', 'TechBusinessDateTime']
BASE_DIR = '/Users/serhiy.ilchyshyngmail.com/PycharmProjects/yuria-farm-inner'


def validate_entity(entity: str, id_col: str, fabric_sep: str = '\t') -> None:
    api = pd.read_csv(f"{BASE_DIR}/{entity}.csv")
    fab = pd.read_csv(f"{BASE_DIR}/{entity}_fabric.csv", sep=fabric_sep)
    fab = fab.drop(columns=[c for c in TECH_COLS if c in fab.columns])

    print(f"=== {entity} ===")
    print(f"API columns:    {list(api.columns)}")
    print(f"Fabric columns: {list(fab.columns)}")
    print(f"API rows: {len(api)}, Fabric rows: {len(fab)}")

    missing_in_fab = [c for c in api.columns if c not in fab.columns]
    missing_in_api = [c for c in fab.columns if c not in api.columns]
    if missing_in_fab:
        print(f"  MISSING in Fabric: {missing_in_fab}")
    if missing_in_api:
        print(f"  EXTRA in Fabric (not in API): {missing_in_api}")

    api = api.set_index(id_col).sort_index()
    fab = fab.set_index(id_col).sort_index()

    api_ids = set(api.index)
    fab_ids = set(fab.index)
    only_api = sorted(api_ids - fab_ids)
    only_fab = sorted(fab_ids - api_ids)
    if only_api:
        print(f"  IDs only in API:    {only_api}")
    if only_fab:
        print(f"  IDs only in Fabric: {only_fab}")

    common_cols = [c for c in api.columns if c in fab.columns]
    common_ids = sorted(api_ids & fab_ids)
    found_diff = False
    for col in common_cols:
        api_col = api.loc[common_ids, col].astype(str).str.strip()
        fab_col = fab.loc[common_ids, col].astype(str).str.strip()
        diffs = api_col[api_col != fab_col]
        if not diffs.empty:
            found_diff = True
            print(f"  Column \'{col}\' differs for {id_col}s: {list(diffs.index)}")
            for rid in diffs.index:
                print(f"    {id_col}={rid}: API={api_col[rid]!r}  Fabric={fab_col[rid]!r}")

    if not found_diff and not only_api and not only_fab and not missing_in_fab and not missing_in_api:
        print("  OK - datasets match.")
    print()


validate_entity(entity='buyers', id_col='buyerid', fabric_sep='\t')
validate_entity(entity='emp',    id_col='empref',  fabric_sep=';')
